# MSA 2025 Phase 2 - Part 1

In [1]:
import sklearn
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Find all variables and understand them

In [ ]:
df = pd.read_csv('store_sales.csv', parse_dates=['Order Date', 'Ship Date'])
#Read the data and convert the Order Date and Ship Date to the datetime type for subsequent time analysis
df.head(10)
#Check the first 10 lines to understand the data structure and sample situation
df.info()
#View data type
df.dtypes
#Basic information of the output dataset: total number of rows and columns, type, missing value situation, memory usage, etc
df.describe(include='all')
#Generate statistical summaries, including descriptive statistics of numerical columns and categorical columns
#This generates key statistics for all numeric columns, including: mean — average value, std — standard deviation, min, max, and percentile ranges.
sales_mean = df['Sales'].mean()
sales_std = df['Sales'].std()
#sales_mean = 249.8...
#sales_std = 503.1...

means = df[numeric_cols].mean()
stds = df[numeric_cols].std()
#Getting stats for all numeric columns

#variable declaration
| Column Name   | Data Type        | Notes                                 |
| ------------- | ---------------- | ------------------------------------- |
| Row ID        | `int`            | Integer row identifier                |
| Order ID      | `object`         | String—order identifier               |
| Order Date    | `datetime`       | Date of order — parsed as datetime    |
| Ship Date     | `datetime`       | Date of shipping — parsed as datetime |
| Ship Mode     | `category`       | Shipping method (e.g. Standard Class) |
| Customer ID   | `object`         | Customer identifier                   |
| Customer Name | `object`         | Customer’s name                       |
| Segment       | `category`       | Customer segment (e.g. Consumer)      |
| Country       | `category`       | Country name                          |
| City          | `category`       | City name                             |
| State         | `category`       | State or province                     |
| Postal Code   | `object`         | Postal code stored as string          |
| Region        | `category`       | Geographic region                     |
| Product ID    | `object`         | Product identifier                    |
| Category      | `category`       | Product category                      |
| Sub-Category  | `category`       | Product sub-category                  |
| Product Name  | `object`         | Name of the product                   |
| Sales         | `float`          | Sales revenue                         |
| Quantity      | `int`            | Number of units sold                  |
| Discount      | `float`          | Discount rate                         |
| Profit        | `float`          | Profit value                          |


## 2. Clean data

In [ ]:
# Missing value handling
df.isnull().sum()
# Count missing values per column
df = df.dropna(axis=0)
# If only a few rows have missing entries, drop them

# For numeric columns, impute missing with median
from sklearn.impute import SimpleImputer
imp = SimpleImputer(strategy='median')
df['Sales'] = imp.fit_transform(df[['Sales']])
#We drop rows with missing values to preserve most of the dataset, and use median imputation to avoid skewing by outliers. This ensures our dataset is complete and ready for modeling.

# Outlier identification and handling
Q1 = df['Profit'].quantile(0.25)
Q3 = df['Profit'].quantile(0.75)
IQR = Q3 - Q1
mask = df['Profit'].between(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
df = df.loc[mask]
#Filtering with IQR filtering removes extreme outliers that could distort the model. We preserve most data while avoiding skew.

upper = df['Profit'].quantile(0.99)
df['Profit'] = np.where(df['Profit'] > upper, upper, df['Profit'])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df['Profit_scaled'] = scaler.fit_transform(df[['Profit']])

# Sample imbalance
from imblearn.over_sampling import SMOTE
X = df.drop(columns='target')
y = df['target']
X_res, y_res = SMOTE().fit_resample(X, y)

# Feature selection
from sklearn.feature_selection import SelectKBest, f_regression
X = df.drop(columns=['Sales', 'Profit'])
y = df['Profit']
best = SelectKBest(f_regression, k=10)
X_new = best.fit_transform(X, y)
selected_cols = X.columns[best.get_support()]

## 3. Visualise data

In [ ]:
#Skewed sales or profit distributions suggest a few high-value transactions.
#Box plots reveal outliers—e.g., extremely high or negative profits.
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

# Histograms for distribution and skewness
df[num_cols].hist(bins=30, figsize=(10,8))
plt.suptitle("Histograms of Numeric Features")
plt.show()

# Box plots for detecting outliers
plt.figure(figsize=(10,6))
sns.boxplot(data=df[num_cols])
plt.title("Boxplot of Numeric Features")
plt.show()

#Summarize and draw the time series trend line by week
df['Week'] = df['Order Date'].dt.to_period('W').apply(lambda r: r.start_time)
weekly = df.groupby('Week')['Sales'].sum().reset_index()

plt.figure(figsize=(12,4))
sns.lineplot(data=weekly, x='Week', y='Sales', marker='o')
plt.title('Weekly Total Sales')
plt.xlabel('Week')
plt.ylabel('Sales')
plt.grid(True)
#Look for peaks (like seasonal spikes) or dips (off-seasons or anomalies).

#To visualize how different segments perform over time:
weekly_region = df.groupby(['Order Week','Region'])['Sales'].sum().reset_index()

plt.figure(figsize=(12,5))
sns.lineplot(data=weekly_region, x='Order Week', y='Sales', hue='Region')
plt.title("Weekly Sales by Region")
plt.xlabel("Week")
plt.ylabel("Sales")
plt.legend(title='Region')
plt.show()
#See which regions lead or lag; identify unusual behavior.

#Use box or violin plots (recommended by the modules) for distribution comparisons:
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x='Region', y='Profit')
plt.title("Profit Distribution by Region")
plt.show()

plt.figure(figsize=(10,6))
sns.violinplot(data=df, x='Category', y='Profit')
plt.title("Profit Distribution by Category")
plt.xticks(rotation=45)
plt.show()
#Compare central tendency and variability across regions/categories.
#Spot which categories have high volatility or negative profits.

#Point plots are ideal for comparing group means with confidence intervals:
plt.figure(figsize=(10,6))
sns.pointplot(data=df, x='Category', y='Profit', hue='Region', capsize=.2)
plt.title("Mean Profit by Category & Region")
plt.xticks(rotation=45)
plt.show()
#Highlights average profitability by category and region, with error bars showing variability.


#Frequency analysis: The bar chart shows the transformed categorical variables
sns.countplot(data=df, x='Region')
sns.countplot(data=df, x='Category')

#Trend comparison after categorical variable conversion
weekly_region = df.groupby(['Week','Region'])['Sales'].sum().reset_index()
sns.lineplot(data=weekly_region, x='Week', y='Sales', hue='Region')
plt.title('Weekly Sales by Region')

#The relationship between numerical values and categories
sns.boxplot(data=df, x='Region', y='Sales')
sns.violinplot(data=df, x='Category', y='Profit')

#Average value estimation graph
sns.pointplot(data=df, x='Category', y='Profit', hue='Region', capsize=.2)


## 4. Identify correlated variables

In [ ]:
#Compute Correlation Matrix
# Select only numerical columns for correlation analysis
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

# Compute Pearson correlation matrix
corr_matrix = df[num_cols].corr()

#Identify Strongly Correlated Pairs
# Identify variable pairs with correlation above 0.8 or below -0.8
strong_pairs = [(i, j, corr_matrix.loc[i,j])
                for i in num_cols for j in num_cols
                if i != j and abs(corr_matrix.loc[i,j]) > 0.8]

#Decide Which Feature to Drop/Keep
#Example: strong correlation detected between Sales and Profit
#Suppose we drop 'Sales' to keep Profit as the target
df_reduced = df.drop(columns=['Sales'])

#Visualize with Heatmap
# Mask to show only half of the symmetric matrix
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(6,4))
sns.heatmap(corr_matrix, mask=mask, annot=True, vmin=-1, vmax=1, cmap='BrBG')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

## 5. Summary